In [19]:
import xarray as xr
import numpy as np
import pandas as pd

# Load the future datasets (2071-2100)
pr_future_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_future585_20712100_annual.nc"
hfls_future_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_future585_20712100_annual.nc"

# Load the historical datasets (1981-2010)
pr_hist_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_present_19812010_annual.nc"  # Update with actual path
hfls_hist_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_present_19812010_annual.nc"  # Update with actual path

# Function to calculate P-E
def calculate_p_minus_e(pr_file, hfls_file):
    ds_pr = xr.open_dataset(pr_file)
    ds_hfls = xr.open_dataset(hfls_file)
    
    # Extract variables
    pr = ds_pr['pr']  # Precipitation (kg m-2 s-1)
    hfls = ds_hfls['hfls']  # Surface upward latent heat flux (W m-2)
    
    # Convert precipitation from kg m-2 s-1 to mm/day
    pr_mm_day = pr * 86400
    
    # Convert latent heat flux to evaporation in mm/day
    L_v = 2.5e6  # J/kg (latent heat of vaporization)
    rho_w = 1000  # kg/m3 (density of water)
    e_mm_day = (hfls / (L_v * rho_w)) * 86400 * 1000
    
    # Calculate P - E
    p_minus_e = pr_mm_day - e_mm_day
    
    # Calculate annual mean for each year, then average over all years
    p_minus_e_annual = p_minus_e.groupby('time.year').mean('time')
    p_minus_e_mean = p_minus_e_annual.mean('year')
    
    # Close datasets
    ds_pr.close()
    ds_hfls.close()
    
    return p_minus_e_mean, ds_pr['lat'].values, ds_pr['lon'].values

# Calculate P-E for both periods
p_minus_e_future, lat, lon = calculate_p_minus_e(pr_future_file, hfls_future_file)
p_minus_e_hist, _, _ = calculate_p_minus_e(pr_hist_file, hfls_hist_file)

# Calculate area weights
weights = np.cos(np.deg2rad(lat))
weights_2d = np.broadcast_to(weights[:, np.newaxis], (len(lat), len(lon)))
weights_da = xr.DataArray(weights_2d, dims=['lat', 'lon'], coords={'lat': lat, 'lon': lon})

# Define regions
lat_bins = [(-60, -35), (-35, -23.5), (-23.5, 23.5), (23.5, 35), (35, 60)]
region_names = ['R01_SM', 'R02_SS', 'R03_Tropics', 'R04_NS', 'R05_NM']

# Calculate area-weighted regional mean for both periods
results = []
for region_name, (lat_min, lat_max) in zip(region_names, lat_bins):
    # Select region for future
    region_data_future = p_minus_e_future.sel(lat=slice(lat_min, lat_max))
    region_weights = weights_da.sel(lat=slice(lat_min, lat_max))
    weighted_mean_future = (region_data_future * region_weights).sum() / region_weights.sum()
    
    # Select region for historical
    region_data_hist = p_minus_e_hist.sel(lat=slice(lat_min, lat_max))
    weighted_mean_hist = (region_data_hist * region_weights).sum() / region_weights.sum()
    
    # Calculate delta
    delta = weighted_mean_future - weighted_mean_hist
    
    results.append({
        'Region': region_name,
        'P_minus_E_1981_2010_mm_day': float(weighted_mean_hist.values),
        'P_minus_E_2071_2100_mm_day': float(weighted_mean_future.values),
        'Delta_mm_day': float(delta.values)
    })

# Create DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv('P_minus_E_delta_CMCC-ESM2.csv', index=False)

print(f"Saved to P_minus_E_delta_MIROC6.csv")
print(f"\n{df}")


Saved to P_minus_E_delta_MIROC6.csv

        Region  P_minus_E_1981_2010_mm_day  P_minus_E_2071_2100_mm_day  \
0       R01_SM                    0.632444                    0.658193   
1       R02_SS                   -1.549127                   -1.797830   
2  R03_Tropics                   -0.219970                   -0.268122   
3       R04_NS                   -0.966551                   -1.029283   
4       R05_NM                    0.631944                    0.705454   

   Delta_mm_day  
0      0.025749  
1     -0.248704  
2     -0.048152  
3     -0.062732  
4      0.073510  


In [13]:
import xarray as xr
import numpy as np
import pandas as pd

# Load the datasets
pr_file = "/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/pr/MIROC6_pr_future585_20712100_annual.nc"
hfls_file = "/Users/dianindrawati/Documents/CMIP6_datasets/MIROC6/hfls/MIROC6_hfls_future585_20712100_annual.nc"

ds_pr = xr.open_dataset(pr_file)
ds_hfls = xr.open_dataset(hfls_file)

# Extract variables
pr = ds_pr['pr']  # Precipitation (kg m-2 s-1)
hfls = ds_hfls['hfls']  # Surface upward latent heat flux (W m-2)

# Convert precipitation from kg m-2 s-1 to mm/day
pr_mm_day = pr * 86400

# Convert latent heat flux to evaporation in mm/day
L_v = 2.5e6  # J/kg (latent heat of vaporization)
rho_w = 1000  # kg/m3 (density of water)
e_mm_day = (hfls / (L_v * rho_w)) * 86400 * 1000

# Calculate P - E
p_minus_e = pr_mm_day - e_mm_day

# Calculate area weights
lat = ds_pr['lat'].values
lon = ds_pr['lon'].values
weights = np.cos(np.deg2rad(lat))
weights_2d = np.broadcast_to(weights[:, np.newaxis], (len(lat), len(lon)))
weights_da = xr.DataArray(weights_2d, dims=['lat', 'lon'], coords={'lat': lat, 'lon': lon})

# Calculate annual mean for each year, then average over all years (2071-2100)
p_minus_e_annual = p_minus_e.groupby('time.year').mean('time')
p_minus_e_mean = p_minus_e_annual.mean('year')

# Define regions
lat_bins = [(-60, -35), (-35, -23.5), (-23.5, 23.5), (23.5, 35), (35, 60)]
region_names = ['R01_SM', 'R02_SS', 'R03_Tropics', 'R04_NS', 'R05_NM']

# Calculate area-weighted regional mean (one value per region)
results = []
for region_name, (lat_min, lat_max) in zip(region_names, lat_bins):
    # Select region
    region_data = p_minus_e_mean.sel(lat=slice(lat_min, lat_max))
    region_weights = weights_da.sel(lat=slice(lat_min, lat_max))
    
    # Calculate area-weighted mean
    weighted_mean = (region_data * region_weights).sum() / region_weights.sum()
    
    results.append({
        'Region': region_name,
        'P_minus_E_mm_day': float(weighted_mean.values)
    })

# Create DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv('mean_P_minus_E_2071_2100_MIROC6.csv', index=False)

print(f"Saved to mean_P_minus_E_2071_2100_CMCC-ESM2.csv")
print(f"\n{df}")

# Close datasets
ds_pr.close()
ds_hfls.close()

Saved to mean_P_minus_E_2071_2100_CMCC-ESM2.csv

        Region  P_minus_E_mm_day
0       R01_SM          0.782276
1       R02_SS         -1.471235
2  R03_Tropics         -0.395580
3       R04_NS         -0.933924
4       R05_NM          0.773260


In [8]:
import xarray as xr
import numpy as np
import pandas as pd

# Load the datasets
pr_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_future585_20712100_annual.nc"
hfls_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_future585_20712100_annual.nc"

ds_pr = xr.open_dataset(pr_file)
ds_hfls = xr.open_dataset(hfls_file)

# Extract variables
pr = ds_pr['pr']  # Precipitation (kg m-2 s-1)
hfls = ds_hfls['hfls']  # Surface upward latent heat flux (W m-2)

# Convert precipitation from kg m-2 s-1 to mm/day
pr_mm_day = pr * 86400

# Convert latent heat flux to evaporation in mm/day
L_v = 2.5e6  # J/kg (latent heat of vaporization)
rho_w = 1000  # kg/m3 (density of water)
e_mm_day = (hfls / (L_v * rho_w)) * 86400 * 1000

# Calculate P - E
p_minus_e = pr_mm_day - e_mm_day

# Calculate area weights
lat = ds_pr['lat'].values
lon = ds_pr['lon'].values
weights = np.cos(np.deg2rad(lat))
weights_2d = np.broadcast_to(weights[:, np.newaxis], (len(lat), len(lon)))
weights_da = xr.DataArray(weights_2d, dims=['lat', 'lon'], coords={'lat': lat, 'lon': lon})

# Calculate annual mean for each year
p_minus_e_annual = p_minus_e.groupby('time.year').mean('time')

# Define regions
lat_bins = [(-60, -35), (-35, -23.5), (-23.5, 23.5), (23.5, 35), (35, 60)]
region_names = ['R01_SM', 'R02_SS', 'R03_Tropics', 'R04_NS', 'R05_NM']

# Calculate area-weighted regional mean for each year
results = []
for year in p_minus_e_annual.year.values:
    year_data = p_minus_e_annual.sel(year=year)
    
    for region_name, (lat_min, lat_max) in zip(region_names, lat_bins):
        # Select region
        region_data = year_data.sel(lat=slice(lat_min, lat_max))
        region_weights = weights_da.sel(lat=slice(lat_min, lat_max))
        
        # Calculate area-weighted mean
        weighted_mean = (region_data * region_weights).sum() / region_weights.sum()
        
        results.append({
            'Year': int(year),
            'Region': region_name,
            'P_minus_E_mm_day': float(weighted_mean.values)
        })

# Create DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv('annual_P_minus_E_2071_2100_CMCC-ESM2.csv', index=False)

print(f"Saved to annual_P_minus_E_2071_2100_CMCC-ESM2.csv")
print(f"Total rows: {len(df)}")
print(f"\nFirst 15 rows:")
print(df.head(15))
print(f"\nLast 15 rows:")
print(df.tail(15))

# Close datasets
ds_pr.close()
ds_hfls.close()

Saved to annual_P_minus_E_2071_2100_CMCC-ESM2.csv
Total rows: 150

First 15 rows:
    Year       Region  P_minus_E_mm_day
0   2071       R01_SM          0.689681
1   2071       R02_SS         -1.741036
2   2071  R03_Tropics         -0.327621
3   2071       R04_NS         -0.806088
4   2071       R05_NM          0.748878
5   2072       R01_SM          0.596135
6   2072       R02_SS         -1.642745
7   2072  R03_Tropics         -0.247530
8   2072       R04_NS         -1.061141
9   2072       R05_NM          0.689192
10  2073       R01_SM          0.621624
11  2073       R02_SS         -1.703132
12  2073  R03_Tropics         -0.276199
13  2073       R04_NS         -0.886785
14  2073       R05_NM          0.708292

Last 15 rows:
     Year       Region  P_minus_E_mm_day
135  2098       R01_SM          0.630934
136  2098       R02_SS         -1.899152
137  2098  R03_Tropics         -0.250148
138  2098       R04_NS         -1.095710
139  2098       R05_NM          0.760726
140  2099       R

In [7]:
import xarray as xr
import numpy as np
import pandas as pd

# Load the datasets
pr_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/pr/CMCC-ESM2_pr_future585_20712100_annual.nc"  # Replace with your precipitation file path
hfls_file = "/Users/dianindrawati/Documents/CMIP6_datasets/CMCC-ESM2/hfls/CMCC-ESM2_hfls_future585_20712100_annual.nc"  # Replace with your evaporation file path

ds_pr = xr.open_dataset(pr_file)
ds_hfls = xr.open_dataset(hfls_file)

# Extract variables
pr = ds_pr['pr']  # Precipitation (kg m-2 s-1)
hfls = ds_hfls['hfls']  # Surface upward latent heat flux (W m-2)

# Convert precipitation from kg m-2 s-1 to mm/day
pr_mm_day = pr * 86400

# Convert latent heat flux to evaporation in mm/day
L_v = 2.5e6  # J/kg (latent heat of vaporization)
rho_w = 1000  # kg/m3 (density of water)
e_mm_day = (hfls / (L_v * rho_w)) * 86400 * 1000

# Calculate P - E
p_minus_e = pr_mm_day - e_mm_day

# Calculate area weights
lat = ds_pr['lat'].values
lon = ds_pr['lon'].values
weights = np.cos(np.deg2rad(lat))
weights_2d = np.broadcast_to(weights[:, np.newaxis], (len(lat), len(lon)))
weights_da = xr.DataArray(weights_2d, dims=['lat', 'lon'], coords={'lat': lat, 'lon': lon})

# Calculate annual mean for each year
p_minus_e_annual = p_minus_e.groupby('time.year').mean('time')

# Calculate area-weighted global mean for each year
results = []
for year in p_minus_e_annual.year.values:
    year_data = p_minus_e_annual.sel(year=year)
    weighted_mean = (year_data * weights_da).sum() / weights_da.sum()
    results.append({
        'Year': int(year),
        'P_minus_E_mm_day': float(weighted_mean.values)
    })

# Create DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv('annual_P_minus_E_2071_2100_CMCC-ESM2.csv', index=False)

print(f"Saved to annual_P_minus_E_2071_2100.csv")
print(f"\n{df}")

# Close datasets
ds_pr.close()
ds_hfls.close()

Saved to annual_P_minus_E_2071_2100.csv

    Year  P_minus_E_mm_day
0   2071          0.000267
1   2072          0.000088
2   2073         -0.004872
3   2074         -0.003669
4   2075         -0.000860
5   2076         -0.000537
6   2077         -0.001106
7   2078         -0.004510
8   2079         -0.005728
9   2080         -0.001129
10  2081          0.000492
11  2082         -0.002646
12  2083         -0.001092
13  2084         -0.003466
14  2085         -0.004298
15  2086         -0.003709
16  2087         -0.001081
17  2088          0.000544
18  2089         -0.003098
19  2090         -0.002997
20  2091         -0.005486
21  2092          0.000607
22  2093         -0.001385
23  2094         -0.003152
24  2095         -0.005079
25  2096         -0.001110
26  2097         -0.000348
27  2098         -0.003565
28  2099         -0.003151
29  2100          0.000903
